**1. Import Libraries and Mount to Drive**

In [ ]:
# enables 4-bit/8-bit quantization and 32-bit optimizers (makes it possble for fine-tuning with less vram)
!pip install -U bitsandbytes

# import libraries
import torch
import bitsandbytes as bnb
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
)
from google.colab import drive
import os
import pandas as pd
from torch.nn.utils.rnn import pad_sequence
import math
import shutil
import time
from transformers import TrainerCallback
from google.colab import files

# mount to personal google drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 53.5 MB/s eta 0:00:00
Mounted at /content/drive


**2. Confirm GPU is Active**

In [ ]:
# finds out if the gpu is available
if not torch.cuda.is_available():
  print("GPU currently not available. Please select the correct runtime type (Runtime → Change runtime type → GPU).")
else:
  print(f"   - GPU detected: {torch.cuda.get_device_name(0)}") # displays the gpu name
  print(f"   - VRAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB") # displays how much memory the gpu has
  print(f"   - PyTorch CUDA version: {torch.version.cuda}") # displays what version of CUDA PyTorch is being used to talk to the GPU

  # tells pytorch to use the GPU
  device = torch.device("cuda")

   - GPU detected: NVIDIA RTX PRO 6000 Blackwell Server Edition
   - VRAM available: 102.0 GB
   - PyTorch CUDA version: 12.8


**3. Load Model + Tokenizer**

In [ ]:
# name of the pre-trained model being load
model_name = "meta-llama/Llama-3.2-3B-Instruct"

# load tokenizer for the model (coverts text into numbers / converts numbers back to text)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token  # llama has no pad token by default so borrow the eos token instead

# load the model in bfloat16 on the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map="cuda",
)

# disable cache to save memory during training
model.config.use_cache = False

# align the model's pad token ID with the tokenizer so loss calculations ignore padded space
model.config.pad_token_id = tokenizer.pad_token_id

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

**4. Load Dataset (Non Mathematical)**

In [ ]:
# folder path to the medical dataset files
folder_path = "/content/drive/MyDrive/Fine-Tuning-Datasets/MedInstruct"

# load the train and val sets for the medical dataset
train_df = pd.read_json(f"{folder_path}/medical_train.jsonl", lines=True)
val_df   = pd.read_json(f"{folder_path}/medical_val.jsonl",   lines=True)

# convert dataset to Hugging face format
train_ds = Dataset.from_pandas(train_df)
val_ds   = Dataset.from_pandas(val_df)

# calculates full length of dataset
medical_ds = len(train_ds) + len(val_ds)

print(f"Medical Dataset Size: {medical_ds}")
print(f"Train Dataset Size: {len(train_ds)}")
print(f"Val Dataset Size: {len(val_ds)}")

Medical Dataset Size: 56335
Train Dataset Size: 53370
Val Dataset Size: 2965


**5. Convert Instruction Format to Llama-3.2-3B-Instruct Chat Templete**

In [ ]:
# formats each dataset row into the Llama 3 chat template
def format_instruction(sample):

    # system prompt left empty because it was adding date infromation the model did not need to learn
    system_prompt = ""

    # get the main question instruction and any extra details input from the data
    user_prompt = sample["instruction"]
    input_text = sample.get("input", "")

    # combines instruction and input together if exists to make one question
    if input_text and input_text.strip():
        user_prompt = f"{user_prompt}\n\n{input_text.strip()}"

    # manually format to Llama 3 chat template  https://www.llama.com/docs/model-cards-and-prompt-formats/llama3_1/
    text = (
      f"<|start_header_id|>system<|end_header_id|>\n\n"
      f"{system_prompt}<|eot_id|>"
      f"<|start_header_id|>user<|end_header_id|>\n\n"
      f"{user_prompt}<|eot_id|>"
      f"<|start_header_id|>assistant<|end_header_id|>\n\n"
      f"{sample['output']}<|eot_id|>"
    )

    return {"text": text} # returns the formated text

# applies the templates to the dataset
train_ds = train_ds.map(format_instruction)
val_ds = val_ds.map(format_instruction)

# confirms that the chat template was applied
print(train_ds[2]["text"])

Map:   0%|          | 0/53370 [00:00<?, ? examples/s]

Map:   0%|          | 0/2965 [00:00<?, ? examples/s]

<|start_header_id|>system<|end_header_id|>

<|eot_id|><|start_header_id|>user<|end_header_id|>

Based on the provided symptom description, identify the possible diagnosis/test to verify it.

A patient reports constant fatigue, dry skin, weight gain, muscle aches, feelings of being cold all the time, an increased sensitivity to cold, and constipation<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Based on the provided symptoms, the possible diagnosis could be hypothyroidism. To verify this diagnosis, a thyroid function test, specifically measuring levels of thyroid-stimulating hormone (TSH) and thyroxine (T4), should be conducted. Hypothyroidism is a condition characterized by an underactive thyroid gland, which leads to decreased production of thyroid hormones.<|eot_id|>


**6. Apply Tokenization**

In [ ]:
# tokenizes the data into numbers
def tokenize(sample):
  tokens = tokenizer(
      sample["text"],
      truncation=True,  # added as a safety net even though this was already caught in preprocessing
      max_length=2048,   # limits input length to 2048 tokens

      padding=False # handeled in custom data collector section 8
  )
  return tokens

# applies the tokenizer to each set
train_ds = train_ds.map(tokenize, batched=True)
val_ds   = val_ds.map(tokenize,   batched=True)

# confirms tokenizer has been applied
print("Successfully applied tokenize")
print("\n")
print(train_ds[4]["input_ids"])
print("\n")
print(tokenizer.convert_ids_to_tokens(train_ds[2]["input_ids"]))

Map:   0%|          | 0/53370 [00:00<?, ? examples/s]

Map:   0%|          | 0/2965 [00:00<?, ? examples/s]

Successfully applied tokenize


[128000, 128006, 9125, 128007, 271, 128009, 128006, 882, 128007, 271, 29401, 1463, 279, 5995, 7296, 61003, 369, 264, 3230, 6593, 1296, 477, 10537, 382, 1182, 32208, 128009, 128006, 78191, 128007, 271, 10438, 264, 19084, 8737, 11, 6978, 2011, 4148, 31817, 323, 9501, 6302, 11, 1778, 439, 3999, 797, 34968, 477, 7013, 75658, 13, 2435, 1253, 387, 4691, 311, 2349, 1139, 264, 8952, 65510, 13, 44430, 1288, 6179, 872, 18985, 9287, 422, 814, 527, 20895, 477, 617, 459, 60754, 311, 80087, 483, 477, 13168, 54631, 11, 439, 1664, 439, 904, 31010, 13, 128009]


['<|begin_of_text|>', '<|start_header_id|>', 'system', '<|end_header_id|>', 'ĊĊ', '<|eot_id|>', '<|start_header_id|>', 'user', '<|end_header_id|>', 'ĊĊ', 'Based', 'Ġon', 'Ġthe', 'Ġprovided', 'Ġsymptom', 'Ġdescription', ',', 'Ġidentify', 'Ġthe', 'Ġpossible', 'Ġdiagnosis', '/test', 'Ġto', 'Ġverify', 'Ġit', '.ĊĊ', 'A', 'Ġpatient', 'Ġreports', 'Ġconstant', 'Ġfatigue', ',', 'Ġdry', 'Ġskin', ',', 'Ġweight', 'Ġgain', ',

**7. Prompt Masking - (Learn from assistant reply tokens only)**

In [ ]:
# sets the assistant header
assistant_header = "<|start_header_id|>assistant"

# mask non-assistant tokens so loss is only calculated on the assistant reply (output)
def mask_labels(sample):

  # grab the tokens and make a copy to edit
  input_ids = sample["input_ids"]
  labels = input_ids.copy()

  # converts assistant header text to token ids
  assistant_token_ids = tokenizer.encode(assistant_header, add_special_tokens=False)

  # stores how long the assistant header is
  header_len = len(assistant_token_ids)
  mask_end = 0

  # find where the assistant response starts in the input
  for i in range(len(input_ids) - header_len):
     if input_ids[i : i + header_len] == assistant_token_ids:
            mask_end = i + header_len + 2
            break

  # hide everything before the assistant reply so the model only learns from it
  labels[:mask_end] = [-100] * mask_end
  return {"labels": labels}

# apply masking to all rows so model only learns from assistant replies
train_ds = train_ds.map(mask_labels, batched=False, remove_columns=["text"])
val_ds   = val_ds.map(mask_labels,   batched=False, remove_columns=["text"])

# confirms prompt masking has been applied
print("Successfully applied prompt masking")
print("\n")
print("Input IDs:", train_ds[4]["input_ids"])
print("\n")
print("Labels:", train_ds[4]["labels"])

Map:   0%|          | 0/53370 [00:00<?, ? examples/s]

Map:   0%|          | 0/2965 [00:00<?, ? examples/s]

Successfully applied prompt masking


Input IDs: [128000, 128006, 9125, 128007, 271, 128009, 128006, 882, 128007, 271, 29401, 1463, 279, 5995, 7296, 61003, 369, 264, 3230, 6593, 1296, 477, 10537, 382, 1182, 32208, 128009, 128006, 78191, 128007, 271, 10438, 264, 19084, 8737, 11, 6978, 2011, 4148, 31817, 323, 9501, 6302, 11, 1778, 439, 3999, 797, 34968, 477, 7013, 75658, 13, 2435, 1253, 387, 4691, 311, 2349, 1139, 264, 8952, 65510, 13, 44430, 1288, 6179, 872, 18985, 9287, 422, 814, 527, 20895, 477, 617, 459, 60754, 311, 80087, 483, 477, 13168, 54631, 11, 439, 1664, 439, 904, 31010, 13, 128009]


Labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 10438, 264, 19084, 8737, 11, 6978, 2011, 4148, 31817, 323, 9501, 6302, 11, 1778, 439, 3999, 797, 34968, 477, 7013, 75658, 13, 2435, 1253, 387, 4691, 311, 2349, 1139, 264, 8952, 65510, 13, 44430, 1288, 617

**8. Masked Label Collactor**

In [ ]:
class MaskedLabelCollator:
  def __init__(self, tokenizer):
    self.tokenizer = tokenizer # used for pad token id later

  # pad all sequences to the longest one in the batch
  def __call__(self, features):

    # convert lists to tensors so the gpu can use them
    input_ids  = [torch.tensor(f["input_ids"])     for f in features]
    labels     = [torch.tensor(f["labels"])         for f in features]
    attn_masks = [torch.tensor(f["attention_mask"]) for f in features]

    # pad everything to same length
    input_ids  = pad_sequence(input_ids,  batch_first=True, padding_value=self.tokenizer.pad_token_id)
    labels     = pad_sequence(labels,     batch_first=True, padding_value=-100)  # -100 = ignored in loss
    attn_masks = pad_sequence(attn_masks, batch_first=True, padding_value=0)

    # return padded sequences to the trainer
    return {
      "input_ids":      input_ids,
      "labels":         labels,
      "attention_mask": attn_masks,
     }

 # set up the collator to pad batches
data_collator = MaskedLabelCollator(tokenizer)

# confirms mask label collator has been applied
print("Successfully applied mask label collator")
print("\n")
print("Input IDs:", train_ds[2]["input_ids"])
print("\n")
print("Labels:", train_ds[2]["labels"])
print("\n")
print("Attention mask:", train_ds[2]["attention_mask"])

Successfully applied mask label collator


Input IDs: [128000, 128006, 9125, 128007, 271, 128009, 128006, 882, 128007, 271, 29815, 389, 279, 3984, 49648, 4096, 11, 10765, 279, 3284, 23842, 12986, 311, 10356, 433, 382, 32, 8893, 6821, 6926, 36709, 11, 9235, 6930, 11, 4785, 8895, 11, 16124, 264, 8696, 11, 16024, 315, 1694, 9439, 682, 279, 892, 11, 459, 7319, 27541, 311, 9439, 11, 323, 738, 49686, 128009, 128006, 78191, 128007, 271, 29815, 389, 279, 3984, 13803, 11, 279, 3284, 23842, 1436, 387, 9950, 29671, 1607, 2191, 13, 2057, 10356, 420, 23842, 11, 264, 54060, 734, 1296, 11, 11951, 30090, 5990, 315, 54060, 5594, 318, 15853, 36908, 320, 51, 8758, 8, 323, 26236, 55889, 483, 320, 51, 19, 705, 1288, 387, 13375, 13, 39515, 29671, 1607, 2191, 374, 264, 3044, 32971, 555, 459, 1234, 3104, 54060, 67169, 11, 902, 11767, 311, 25983, 5788, 315, 54060, 44315, 13, 128009]


Labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100

**9. Full Fine-Tuning Configuration**

In [ ]:
# full fine-tuning trains all model parameters so requires_grad is set to true for all of them
for param in model.parameters():
    param.requires_grad = True

# confirms how many parameters are being trained vs frozen
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"trainable params: {trainable_params:,} || all params: {total_params:,} || trainable%: {100 * trainable_params / total_params:.2f}%")

trainable params: 3,212,749,824 || all params: 3,212,749,824 || trainable%: 100.00%


**10. Training Arguments**

In [ ]:
training_args = TrainingArguments(
    output_dir="./medical_llama3_full_output", # folder were model checkpoints are saved

    # training batch size 8 x 4 = 32
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,

    per_device_eval_batch_size=4, # eval batch size 4

    gradient_checkpointing=True, # saves memory but makes training slower

    optim="paged_adamw_8bit", # optimizor to use

    learning_rate=2e-5, # learning rate of the model
    lr_scheduler_type="cosine", # slowly reduce learning rate over training

    warmup_steps=200, # slowly warms up learning rate

    weight_decay=0.05, # reduce overfitting by shrinking large weights
    max_grad_norm=1.0,  # clip gradients to stop training going unstable

    num_train_epochs=3, # times model see training dataset

    # log every 20 steps, evaluate every 200 steps
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=200,

    # save checkpoint every 200 steps, keep only the 2 best, load best at end
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,

    # use bfloat16
    bf16=True,
    fp16=False,

    # speed up data loading
    dataloader_num_workers=4,
    dataloader_pin_memory=True,

    remove_unused_columns=False, # keep all columns including custom ones
    seed=42, # ensures results will be the same if run again
    data_seed=42, # ensures results are replicable
    report_to="none" # disable logging
)

**11. Trainer**

In [ ]:
# set up the trainer with model, data and training config
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
)

**12. Tranning**

In [ ]:
# callback to track best checkpoint time
class BestCheckpointCallback(TrainerCallback):
    def __init__(self):
        self.best_checkpoint_time = None
        self.last_best_checkpoint = None

    def on_evaluate(self, args, state, control, **kwargs):
        # if best checkpoint changes record time
        if state.best_model_checkpoint != self.last_best_checkpoint:
            self.last_best_checkpoint = state.best_model_checkpoint
            self.best_checkpoint_time = time.ctime()

# creates the callback
callback = BestCheckpointCallback()
trainer.add_callback(callback)

# store start time
train_start_time = time.time()
print(f"\nStarting full fine-tuning at {time.ctime(train_start_time)}\n")

# train
trainer.train()

# store end time
train_end_time = time.time()

# print summary (clean output)
print("\nTraining completed")
print(f"   Finished at:           {time.ctime(train_end_time)}")
print(f"   Total steps:           {trainer.state.global_step:,}")
print(f"   Best checkpoint:       {trainer.state.best_model_checkpoint}")
print(f"   Best checkpoint time:  {callback.best_checkpoint_time}")
print(f"   Total training time:   {(train_end_time - train_start_time)/60:.2f} minutes")


Starting full fine-tuning at Mon Apr 13 18:50:07 2026



Step,Training Loss,Validation Loss
200,0.794597,0.789823
400,0.757409,0.765884
600,0.759151,0.753288
800,0.752556,0.743325
1000,0.745110,0.735417
1200,0.729384,0.728455
1400,0.742605,0.724381
1600,0.696805,0.719643
1800,0.566420,0.730636
2000,0.582458,0.727605


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].



Training completed
   Finished at:           Mon Apr 13 21:29:27 2026
   Total steps:           5,004
   Best checkpoint:       ./medical_llama3_full_output/checkpoint-3000
   Best checkpoint time:  Mon Apr 13 20:28:52 2026
   Total training time:   159.33 minutes


**14. Save Full Fine-Tuning to Google Drive**

In [ ]:
# saves the full fine-tuned model weights and tokenizer
model.save_pretrained("medical_llama3_full_model")
tokenizer.save_pretrained("medical_llama3_full_model")

!zip -r medical_llama3_full_model.zip ./medical_llama3_full_model

# folder path in google drive
folder_path = "/content/drive/MyDrive/Fine-Tuning-Notebooks/Full-Fine-Tuning/Llama-3.2-3B-Instruct/Medical"

# create folder if it doesnt exist
os.makedirs(folder_path, exist_ok=True)

# move zip file into google drive folder
shutil.move("medical_llama3_full_model.zip", f"{folder_path}/medical_llama3_full_model.zip")

# confirm saved
print("Model zip saved to Google Drive")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: medical_llama3_full_model/ (stored 0%)
  adding: medical_llama3_full_model/config.json (deflated 53%)
  adding: medical_llama3_full_model/model.safetensors (deflated 21%)
  adding: medical_llama3_full_model/tokenizer.json (deflated 85%)
  adding: medical_llama3_full_model/chat_template.jinja (deflated 71%)
  adding: medical_llama3_full_model/generation_config.json (deflated 33%)
  adding: medical_llama3_full_model/tokenizer_config.json (deflated 43%)
Model zip saved to Google Drive
